# CNN vs Linear Regression — Head-to-Head Defense

**BioITWorld talk — Notebook 1 of 3.**

This notebook puts the CNN-with-autoencoder result side-by-side with the "pedestrian"
linear-regression baseline that Richard Scheuermann favors, on equal cross-validation
footing. Goal: rigorously test whether the CNN's MSE advantage is real, where it comes
from, and how much of it is closed by giving a linear model richer features (the new
"Jeff" panel).

**Models compared:** OLS (per-study, federated, pooled) · Ridge (already in repo) ·
CNN (3×3 Conv → Dense, retrained per CV fold under matched protocol).

**Two feature panels:**

- **Panel A (legacy, N=150 × 9 features)** — same inputs the existing CNN was trained on.
  5 autoantibodies (MIAA, GAD65, IA2IC, ICA, ZNT8) + 3 age-group dummies + Sex.
- **Panel B (Jeff-extended, N=98 × 23 features, SDY524+569+1737)** — adds BMI, height,
  weight, exact age, disease duration since T1D diagnosis, race, ethnicity, treatment
  cohort. SDY797 has no Jeff data, so excluded from Panel B.

**Evaluation protocol:** 5-fold CV stratified by `Study`. Bootstrap 95% CIs on every
metric (1,000 resamples by default; toggleable via `N_BOOT`). Paired tests on per-fold
MSE between models.

**Outputs:** `results/linear_vs_cnn_*.csv` and `figures/linear_vs_cnn_*.pdf`.


## 1. Setup and data load

In [ ]:
from __future__ import annotations
import sys, os, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd()
# Notebook is in ipynb/; modules/ is at repo root.
if (REPO / "modules").exists():
    sys.path.insert(0, str(REPO / "modules"))
elif (REPO.parent / "modules").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "modules"))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from scipy import stats

import oadr_data as od

RNG_SEED = 42
N_BOOT = 1000
N_FOLDS = 5
N_PERM = 1000
np.random.seed(RNG_SEED)

(REPO / "results").mkdir(exist_ok=True)
(REPO / "figures").mkdir(exist_ok=True)
print("Repo:", REPO)


In [ ]:
# Load both feature panels
A = od.load_panel_a_all()
Xa, ya, fa = od.panel_a_design_matrix(A)
print(f"Panel A: N={len(A)}  features={len(fa)}")
print(A.groupby("Study").size().to_string())

B = od.load_panel_b_all()
Xb, yb, fb = od.panel_b_design_matrix(B)
print(f"\nPanel B: N={len(B)}  features={len(fb)}")
print(B.groupby("Study").size().to_string())


## 2. Cross-validation harness

Single helper that runs any sklearn-compatible regressor under the project's
matched protocol: stratified-by-`Study` 5-fold CV, returns per-fold MSE and
out-of-fold predictions for every subject.


In [ ]:
def cv_per_fold_mse(model_factory, X, y, study_labels, n_splits=N_FOLDS, seed=RNG_SEED):
    """Run stratified-by-study k-fold CV. Returns (per-fold MSE array, oof predictions array)."""
    X = np.asarray(X)
    y = np.asarray(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full_like(y, np.nan, dtype=float)
    fold_mse = []
    for fold_idx, (tr, te) in enumerate(skf.split(X, study_labels)):
        m = model_factory()
        m.fit(X[tr], y[tr])
        yhat = m.predict(X[te])
        oof[te] = yhat
        fold_mse.append(mean_squared_error(y[te], yhat))
    return np.asarray(fold_mse), oof


def boot_ci(values, n_boot=N_BOOT, alpha=0.05, seed=RNG_SEED):
    """Percentile bootstrap CI on the mean of `values`."""
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    boots = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return float(values.mean()), float(lo), float(hi)


## 3. Local OLS per study (Panel A)

OLS fit independently per study. This is the strict per-study analogue of what
the existing per-study Ridge / RF / CNN notebooks produce, so the row-by-row
numbers can be dropped directly into the head-to-head table.


In [ ]:
local_rows = []
for study in od.PANEL_A_STUDIES:
    sub = A[A.Study == study]
    if len(sub) < 6:
        print(f"{study}: N={len(sub)} too small for stratified 5-fold; using LOO MSE instead")
        Xs = sub[fa].values
        ys = sub[od.PANEL_A_TARGET].values
        ypred = np.empty_like(ys)
        for i in range(len(ys)):
            mask = np.ones(len(ys), dtype=bool); mask[i] = False
            m = LinearRegression().fit(Xs[mask], ys[mask])
            ypred[i] = m.predict(Xs[i:i+1])[0]
        mse = mean_squared_error(ys, ypred)
        r2 = 1 - np.sum((ys - ypred) ** 2) / np.sum((ys - ys.mean()) ** 2)
        local_rows.append({"Study": study, "N": len(sub), "Protocol": "LOO",
                           "MSE": mse, "R2_oof": r2})
    else:
        Xs = sub[fa].values
        ys = sub[od.PANEL_A_TARGET].values
        skf = StratifiedKFold(n_splits=min(5, len(sub) // 2), shuffle=True, random_state=RNG_SEED)
        # Stratify within-study by sex as a proxy (no external study labels here)
        strata = sub["Sex"].astype(int).values
        ypred = np.empty_like(ys)
        for tr, te in skf.split(Xs, strata):
            m = LinearRegression().fit(Xs[tr], ys[tr])
            ypred[te] = m.predict(Xs[te])
        mse = mean_squared_error(ys, ypred)
        r2 = 1 - np.sum((ys - ypred) ** 2) / np.sum((ys - ys.mean()) ** 2)
        local_rows.append({"Study": study, "N": len(sub), "Protocol": f"{skf.n_splits}-fold",
                           "MSE": mse, "R2_oof": r2})

local_df = pd.DataFrame(local_rows)
print(local_df.to_string(index=False))
local_df.to_csv("results/linear_vs_cnn_local_ols.csv", index=False)


## 4. Federated OLS (median + IQR aggregation, Panel A)

Mirrors the federated-Ridge convention from `RidgeRegressionAnalysisBaselineCPeptideAUC.ipynb`:
fit OLS per study, take the **median coefficient across studies** as the federated estimate.
Reports IQR for each coefficient. Evaluated by applying the median model to each study's
held-out half.


In [ ]:
# Per-study MinMax scaling is required: SDY797 codes autoantibodies as 0/1 while
# the others are continuous (GAD65 up to ~981). Without scaling, median-aggregating
# raw coefficients is meaningless. Each study's model is fit on its own
# scaled-to-[0,1] features; predictions on study X re-use X's own scaler.
study_scalers = {}
fed_coefs, fed_intercepts = {}, []
for study in od.PANEL_A_STUDIES:
    sub = A[A.Study == study]
    sc = MinMaxScaler().fit(sub[fa].values)
    Xs = sc.transform(sub[fa].values)
    ys = sub[od.PANEL_A_TARGET].values
    m = LinearRegression().fit(Xs, ys)
    study_scalers[study] = sc
    fed_coefs[study] = m.coef_
    fed_intercepts.append(m.intercept_)

coef_mat = np.vstack([fed_coefs[s] for s in od.PANEL_A_STUDIES])
median_coef = np.median(coef_mat, axis=0)
q25, q75 = np.percentile(coef_mat, [25, 75], axis=0)
median_intercept = float(np.median(fed_intercepts))

fed_summary = pd.DataFrame({
    "feature": fa,
    "median_beta_scaled": median_coef,
    "iqr_low": q25,
    "iqr_high": q75,
})
print(fed_summary.to_string(index=False))
fed_summary.to_csv("results/linear_vs_cnn_federated_ols_coefs.csv", index=False)


def federated_predict(X_raw, scaler):
    """Apply the scaler trained on the target study, then the federated median model."""
    return scaler.transform(X_raw) @ median_coef + median_intercept


fed_eval = []
for study in od.PANEL_A_STUDIES:
    sub = A[A.Study == study]
    yhat = federated_predict(sub[fa].values, study_scalers[study])
    fed_eval.append({"Study": study, "N": len(sub),
                     "MSE_federated": mean_squared_error(sub[od.PANEL_A_TARGET], yhat)})
fed_eval_df = pd.DataFrame(fed_eval)
print()
print(fed_eval_df.to_string(index=False))
fed_eval_df.to_csv("results/linear_vs_cnn_federated_ols_eval.csv", index=False)


## 5. Pooled OLS / Ridge under stratified-by-Study 5-fold CV (Panel A)

This is the modern footing for the head-to-head: pooled fit, stratified-by-Study CV,
per-fold MSE reported with bootstrap 95% CI.


In [ ]:
from sklearn.pipeline import Pipeline

results_panelA = {}

def make_ols():
    return Pipeline([("sc", MinMaxScaler()), ("m", LinearRegression())])

def make_ridge(alpha=0.001):
    return Pipeline([("sc", MinMaxScaler()), ("m", Ridge(alpha=alpha))])


ols_mse, ols_oof = cv_per_fold_mse(make_ols, Xa.values, ya.values, A["Study"].values)
results_panelA["OLS"] = {"fold_mse": ols_mse, "oof": ols_oof}
print(f"OLS   per-fold MSE: {ols_mse.round(3)}  mean={ols_mse.mean():.3f}")

ridge_mse, ridge_oof = cv_per_fold_mse(lambda: make_ridge(0.001), Xa.values, ya.values, A["Study"].values)
results_panelA["Ridge"] = {"fold_mse": ridge_mse, "oof": ridge_oof}
print(f"Ridge per-fold MSE: {ridge_mse.round(3)}  mean={ridge_mse.mean():.3f}")


## 6. CNN under matched protocol (Panel A, retrained per fold)

Refit the published 3×3 CNN architecture (Conv2D(16, 3×3) → Flatten → Dense(32, relu)
→ Dense(1)) on each fold's training set with the same MinMax scaling and the same
hyperparameters used in `CNN_Local_noAutoencoders_3x3.ipynb`. This is the
apples-to-apples comparison the existing notebooks were missing.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(RNG_SEED)


def build_cnn():
    m = models.Sequential([
        layers.Input(shape=(3, 3, 1)),
        layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
        layers.Flatten(),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer="adam", loss="mse")
    return m


def cnn_cv_per_fold_mse(X, y, study_labels, n_splits=N_FOLDS, epochs=100, batch=8, seed=RNG_SEED):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full_like(y, np.nan, dtype=float)
    fold_mse = []
    for fold_idx, (tr, te) in enumerate(skf.split(X, study_labels)):
        sc = MinMaxScaler().fit(X[tr])
        Xtr = sc.transform(X[tr]).reshape(-1, 3, 3, 1)
        Xte = sc.transform(X[te]).reshape(-1, 3, 3, 1)
        tf.keras.utils.set_random_seed(seed + fold_idx)
        m = build_cnn()
        m.fit(Xtr, y[tr], epochs=epochs, batch_size=batch, verbose=0)
        yhat = m.predict(Xte, verbose=0).ravel()
        oof[te] = yhat
        fold_mse.append(mean_squared_error(y[te], yhat))
        print(f"  CNN fold {fold_idx + 1}/{n_splits}  N_test={len(te)}  MSE={fold_mse[-1]:.3f}")
    return np.asarray(fold_mse), oof


print("Training CNN under 5-fold-by-Study CV (Panel A)...")
cnn_mse, cnn_oof = cnn_cv_per_fold_mse(Xa.values, ya.values, A["Study"].values)
results_panelA["CNN"] = {"fold_mse": cnn_mse, "oof": cnn_oof}
print(f"\nCNN  per-fold MSE: {cnn_mse.round(3)}  mean={cnn_mse.mean():.3f}")


In [ ]:
# Persist per-fold MSE table for Panel A
panelA_mse_table = pd.DataFrame({m: results_panelA[m]["fold_mse"] for m in results_panelA})
panelA_mse_table.index.name = "fold"
print(panelA_mse_table)

means_cis = []
for model in results_panelA:
    mu, lo, hi = boot_ci(results_panelA[model]["fold_mse"])
    means_cis.append({"Model": model, "MSE_mean": mu, "MSE_ci_lo": lo, "MSE_ci_hi": hi,
                      "Panel": "A"})
panelA_summary = pd.DataFrame(means_cis)
print()
print(panelA_summary.to_string(index=False))
panelA_summary.to_csv("results/linear_vs_cnn_panelA_summary.csv", index=False)
panelA_mse_table.to_csv("results/linear_vs_cnn_panelA_per_fold_mse.csv")


## 7. Statistical test of model difference (paired)

Per-fold MSE is paired across models (same training/test indices). Use a paired
**Wilcoxon signed-rank test** (k=5 folds is too small to trust a t-test
distributional assumption). Report Cohen's d on the MSE differences as the effect
size. This is the rigorous version of "is the CNN really better than OLS?"


In [ ]:
def cohen_d(a, b):
    diffs = np.asarray(a) - np.asarray(b)
    if diffs.std(ddof=1) == 0:
        return float("nan")
    return float(diffs.mean() / diffs.std(ddof=1))


pair_results = []
for ref in ["OLS", "Ridge"]:
    for model in results_panelA:
        if model == ref:
            continue
        a = results_panelA[ref]["fold_mse"]
        b = results_panelA[model]["fold_mse"]
        try:
            stat, p = stats.wilcoxon(a, b, alternative="greater", zero_method="zsplit")
        except ValueError:
            stat, p = float("nan"), float("nan")
        d = cohen_d(a, b)
        pair_results.append({
            "panel": "A", "model_A": ref, "model_B": model,
            "MSE_A_mean": float(np.mean(a)), "MSE_B_mean": float(np.mean(b)),
            "delta_MSE": float(np.mean(a) - np.mean(b)),
            "wilcoxon_W": float(stat), "p_one_sided": float(p),
            "cohens_d": d,
        })

pair_df = pd.DataFrame(pair_results)
print(pair_df.to_string(index=False))
pair_df.to_csv("results/linear_vs_cnn_paired_tests.csv", index=False)


## 8. Permutation test (Panel A)

Shuffle `y` `N_PERM` times, refit each model under the same CV protocol, build the
null MSE distribution. The position of the observed MSE in the null tells us whether
each model is using genuine signal vs noise. CNN should sit further to the left of
its null than OLS does — that's the visual argument that CNN finds non-linear signal
the linear model can't see.

> Heads-up: the CNN permutation loop is the slowest cell in this notebook.
> Adjust `N_PERM` (default 1000) downward to ~200 for quick iteration; bump back up
> for the final talk run. CNN permutation is gated behind `RUN_CNN_PERM`.


In [ ]:
RUN_CNN_PERM = False  # flip on for the talk-ready run
N_PERM_FAST = 200     # used for OLS / Ridge

def perm_null_mse(model_factory, X, y, study_labels, n_perm=N_PERM_FAST, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        fold_mse, _ = cv_per_fold_mse(model_factory, X, y_perm, study_labels)
        null.append(fold_mse.mean())
    return np.asarray(null)


print("Computing null MSE distributions for OLS and Ridge (Panel A)...")
ols_null  = perm_null_mse(make_ols, Xa.values, ya.values, A["Study"].values)
ridge_null = perm_null_mse(lambda: make_ridge(0.001), Xa.values, ya.values, A["Study"].values)

null_dists = {"OLS": ols_null, "Ridge": ridge_null}

if RUN_CNN_PERM:
    print("Computing CNN null (this is slow)...")
    cnn_null = []
    rng = np.random.default_rng(RNG_SEED)
    for i in range(N_PERM_FAST):
        y_perm = rng.permutation(ya.values)
        fold_mse, _ = cnn_cv_per_fold_mse(Xa.values, y_perm, A["Study"].values)
        cnn_null.append(fold_mse.mean())
        if (i + 1) % 25 == 0:
            print(f"  perm {i+1}/{N_PERM_FAST}")
    null_dists["CNN"] = np.asarray(cnn_null)

obs = {m: results_panelA[m]["fold_mse"].mean() for m in null_dists}
perm_rows = [{"model": m,
              "obs_MSE": obs[m],
              "null_MSE_median": float(np.median(null_dists[m])),
              "null_MSE_p2.5": float(np.percentile(null_dists[m], 2.5)),
              "null_MSE_p97.5": float(np.percentile(null_dists[m], 97.5)),
              "p_value_left_tail": float((null_dists[m] <= obs[m]).mean()),
              "n_perm": len(null_dists[m])}
             for m in null_dists]
perm_df = pd.DataFrame(perm_rows)
print(perm_df.to_string(index=False))
perm_df.to_csv("results/linear_vs_cnn_permutation.csv", index=False)


## 9. Learning curves (Panel A)

MSE vs training-set size. Expect OLS to plateau early; the CNN trajectory tells the
"more data → better" story that motivates the federated extension. Bootstrap CIs at
each training size.


In [ ]:
from sklearn.model_selection import KFold

def manual_learning_curve(model_factory, X, y, train_sizes_frac, n_runs=10, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    n = len(X)
    out = {}
    for frac in train_sizes_frac:
        n_train = max(int(frac * n * 0.8), 5)
        scores = []
        for _ in range(n_runs):
            idx = rng.permutation(n)
            tr, te = idx[:n_train], idx[n_train:]
            m = model_factory()
            m.fit(X[tr], y[tr])
            scores.append(mean_squared_error(y[te], m.predict(X[te])))
        out[frac] = scores
    return out

train_fracs = [0.2, 0.4, 0.6, 0.8, 1.0]
ols_lc   = manual_learning_curve(make_ols, Xa.values, ya.values, train_fracs)
ridge_lc = manual_learning_curve(lambda: make_ridge(0.001), Xa.values, ya.values, train_fracs)

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, lc, color in [("OLS", ols_lc, "C0"), ("Ridge α=0.001", ridge_lc, "C1")]:
    means = [np.mean(lc[f]) for f in train_fracs]
    los   = [np.percentile(lc[f], 2.5) for f in train_fracs]
    his   = [np.percentile(lc[f], 97.5) for f in train_fracs]
    ax.plot(train_fracs, means, marker="o", label=name, color=color)
    ax.fill_between(train_fracs, los, his, alpha=0.15, color=color)

ax.set_xlabel("Training fraction (of N=150)")
ax.set_ylabel("Test MSE (log AUC)")
ax.set_title("Learning curves — linear baselines (Panel A)")
ax.legend()
fig.tight_layout()
fig.savefig("figures/learning_curves.pdf", dpi=300)
plt.show()


## 10. Residual diagnostics (OLS, Panel A)

Standard linear-regression sanity checks: Q-Q plot, residuals vs fitted, residuals
vs Study. Visible Study effects → motivates the federated / mixed-effects / non-linear
extensions. The OLS-cant-see-this story is the one a critic of the CNN choice should
have to confront.


In [ ]:
ols_full = LinearRegression().fit(Xa.values, ya.values)
yhat = ols_full.predict(Xa.values)
resid = ya.values - yhat

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Q-Q
sm.qqplot(resid, line="s", ax=axes[0])
axes[0].set_title("Q–Q plot of residuals")

# Residuals vs fitted
axes[1].scatter(yhat, resid, alpha=0.6)
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_xlabel("Fitted")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs fitted")

# Residuals vs Study
study_codes = pd.Categorical(A["Study"].values)
axes[2].boxplot([resid[A.Study == s] for s in od.PANEL_A_STUDIES], labels=od.PANEL_A_STUDIES)
axes[2].axhline(0, color="k", lw=0.5)
axes[2].set_title("Residuals by Study")
axes[2].set_ylabel("Residual")

fig.tight_layout()
fig.savefig("figures/residual_diagnostics.pdf", dpi=300)
plt.show()

# Quantify Study effect on residuals
F, p = stats.f_oneway(*[resid[A.Study == s] for s in od.PANEL_A_STUDIES])
print(f"One-way ANOVA on residuals by Study: F={F:.3f}, p={p:.4f}")


## 11. SDY797 stratified analysis (Panel A)

SDY797 is the only study with binary autoantibody encoding; mixing it with the
continuous-measurement studies should hurt a linear model more than it hurts the
CNN. Run the head-to-head with vs without SDY797 to quantify how much of the linear
model's MSE comes from this encoding mismatch.


In [ ]:
A_no797 = A[A.Study != "SDY797"].reset_index(drop=True)
Xa_no797 = A_no797[fa].values
ya_no797 = A_no797[od.PANEL_A_TARGET].values
study_no797 = A_no797["Study"].values

ols_no797_mse, _   = cv_per_fold_mse(make_ols, Xa_no797, ya_no797, study_no797)
ridge_no797_mse, _ = cv_per_fold_mse(lambda: make_ridge(0.001), Xa_no797, ya_no797, study_no797)

print("Without SDY797 (N=", len(A_no797), "):")
print(f"  OLS   per-fold MSE: {ols_no797_mse.round(3)}  mean={ols_no797_mse.mean():.3f}")
print(f"  Ridge per-fold MSE: {ridge_no797_mse.round(3)}  mean={ridge_no797_mse.mean():.3f}")

cnn_no797_mse, _ = cnn_cv_per_fold_mse(Xa_no797, ya_no797, study_no797)
print(f"  CNN   per-fold MSE: {cnn_no797_mse.round(3)}  mean={cnn_no797_mse.mean():.3f}")

sdy797_rows = []
for name, full_mse, drop_mse in [
    ("OLS",   results_panelA["OLS"]["fold_mse"],   ols_no797_mse),
    ("Ridge", results_panelA["Ridge"]["fold_mse"], ridge_no797_mse),
    ("CNN",   results_panelA["CNN"]["fold_mse"],   cnn_no797_mse),
]:
    sdy797_rows.append({
        "model": name,
        "MSE_with_SDY797": float(full_mse.mean()),
        "MSE_without_SDY797": float(drop_mse.mean()),
        "delta": float(full_mse.mean() - drop_mse.mean()),
    })
sdy797_df = pd.DataFrame(sdy797_rows)
print()
print(sdy797_df.to_string(index=False))
sdy797_df.to_csv("results/linear_vs_cnn_sdy797_sensitivity.csv", index=False)


## 12. Panel A vs Panel B — features vs complexity

The Scheuermann question, sharp version: **how much of the CNN's edge over OLS is
"non-linear modelling" vs "the CNN happens to use a richer feature set"?**

Run OLS and Ridge on both panels (Panel B's richer N=98 set covers 3 studies).
Restrict Panel A to the same 3 studies for a matched-N comparison. If OLS on Panel B
closes most of the CNN gap, Scheuermann's argument has support — the answer was
"feed the linear model better features," not "go non-linear." If it doesn't,
the CNN-defense argument is concrete.


In [ ]:
B_studies = od.PANEL_B_STUDIES  # 524, 569, 1737
A_matched = A[A.Study.isin(B_studies)].reset_index(drop=True)
Xa_m = A_matched[fa].values
ya_m = A_matched[od.PANEL_A_TARGET].values
study_a_m = A_matched["Study"].values
print(f"Matched Panel A (3 studies): N={len(A_matched)}")
print(f"Panel B               : N={len(B)}")

# Panel A (matched)
ols_aM, _   = cv_per_fold_mse(make_ols,                 Xa_m, ya_m, study_a_m)
ridge_aM, _ = cv_per_fold_mse(lambda: make_ridge(0.001), Xa_m, ya_m, study_a_m)
cnn_aM, _   = cnn_cv_per_fold_mse(Xa_m, ya_m, study_a_m)

# Panel B (more features → use a slightly stronger ridge default)
ols_b, _   = cv_per_fold_mse(make_ols,                 Xb.values, yb.values, B["Study"].values)
ridge_b, _ = cv_per_fold_mse(lambda: make_ridge(1.0),  Xb.values, yb.values, B["Study"].values)

panelB_rows = []
for name, ma, mb in [
    ("OLS",   ols_aM,   ols_b),
    ("Ridge", ridge_aM, ridge_b),
    ("CNN",   cnn_aM,   None),
]:
    row = {"Model": name,
           "Panel_A_matched_MSE": float(ma.mean()),
           "Panel_A_matched_CI": list(boot_ci(ma)[1:]),
           "Panel_B_MSE": float(mb.mean()) if mb is not None else None,
           "Panel_B_CI": list(boot_ci(mb)[1:]) if mb is not None else None,
           "Delta_AtoB": float(ma.mean() - mb.mean()) if mb is not None else None}
    panelB_rows.append(row)

panelB_df = pd.DataFrame(panelB_rows)
print()
print(panelB_df.to_string(index=False))
panelB_df.to_json("results/linear_vs_cnn_panelA_vs_panelB.json", orient="records", indent=2)


## 13. Talk-ready summary figure

One figure per panel: forest plot of model MSE with 95% bootstrap CI bars, the
single chart the talk methods slide hangs on.


In [ ]:
def forest_plot(rows, title, savepath):
    fig, ax = plt.subplots(figsize=(7, 0.6 + 0.5 * len(rows)))
    y_pos = np.arange(len(rows))[::-1]
    means = [r["mean"] for r in rows]
    los = [r["lo"] for r in rows]
    his = [r["hi"] for r in rows]
    err = [[m - lo for m, lo in zip(means, los)],
           [hi - m for hi, m in zip(his, means)]]
    ax.errorbar(means, y_pos, xerr=err, fmt="o", color="C0", capsize=4)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([r["label"] for r in rows])
    ax.set_xlabel("Test MSE (log AUC), 5-fold CV ± 95% bootstrap CI")
    ax.set_title(title)
    ax.grid(True, axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(savepath, dpi=300)
    plt.show()
    return fig


# Panel A (4 studies, N=150)
rowsA = []
for m in ("OLS", "Ridge", "CNN"):
    mu, lo, hi = boot_ci(results_panelA[m]["fold_mse"])
    rowsA.append({"label": m, "mean": mu, "lo": lo, "hi": hi})
forest_plot(rowsA, "Panel A — N=150, 4 studies, 9 features", "figures/linear_vs_cnn_forest_panelA.pdf")

# Panel A matched + B
rowsAB = []
for name, mset, label in [
    ("OLS",   ols_aM,   "OLS  · Panel A (matched, N=98)"),
    ("Ridge", ridge_aM, "Ridge · Panel A (matched, N=98)"),
    ("CNN",   cnn_aM,   "CNN  · Panel A (matched, N=98)"),
    ("OLS",   ols_b,    "OLS  · Panel B (N=98, 23 features)"),
    ("Ridge", ridge_b,  "Ridge · Panel B (N=98, 23 features)"),
]:
    mu, lo, hi = boot_ci(mset)
    rowsAB.append({"label": label, "mean": mu, "lo": lo, "hi": hi})
forest_plot(rowsAB, "Panel A (matched) vs Panel B (Jeff-extended)", "figures/linear_vs_cnn_forest_panelAB.pdf")


## 14. What this notebook produced

CSVs in `results/`:

- `linear_vs_cnn_local_ols.csv` — per-study OLS performance
- `linear_vs_cnn_federated_ols_coefs.csv` — federated (median) coefficients with IQR
- `linear_vs_cnn_federated_ols_eval.csv` — federated model on each study
- `linear_vs_cnn_panelA_per_fold_mse.csv` — fold-level MSE per model (Panel A)
- `linear_vs_cnn_panelA_summary.csv` — model means + 95% bootstrap CIs (Panel A)
- `linear_vs_cnn_paired_tests.csv` — Wilcoxon paired tests + Cohen's d
- `linear_vs_cnn_permutation.csv` — observed vs null MSE
- `linear_vs_cnn_sdy797_sensitivity.csv` — with vs without SDY797
- `linear_vs_cnn_panelA_vs_panelB.json` — feature-richness vs model-complexity comparison

Figures in `figures/`:

- `learning_curves.pdf`
- `residual_diagnostics.pdf`
- `linear_vs_cnn_forest_panelA.pdf`
- `linear_vs_cnn_forest_panelAB.pdf`

Continue to **Notebook 2** (`LASSO_ElasticNet_Autoantibody_CPeptide.ipynb`) for the
LASSO iteration, then **Notebook 3** (`EffectSizes_and_PowerAnalysis_Extended.ipynb`)
for effect-size and power layering.
